# Python Async Basics

**Goal:** async/await, asyncio, and the event loop — how Python's cooperative concurrency differs from Go's goroutines.

**Prereq:** Complete [02_oop_patterns.ipynb](./02_oop_patterns.ipynb) first.

## The Mental Model Shift

**Go/TS comparison:** Go goroutines are **preemptive** — the runtime switches between them automatically. Python async is **cooperative** — your code must explicitly `await` to let other tasks run. It's like TS `async/await` with `asyncio` as the event loop (similar to Node's event loop). Only ONE thing runs at a time.

In [ ]:
import asyncio
import time

# Basic coroutine — async def creates a coroutine function
async def greet(name, delay):
    await asyncio.sleep(delay)  # Non-blocking sleep
    return f"Hello, {name}!"

# In Jupyter, you can await directly (the notebook has a running event loop)
result = await greet("Kyle", 1)
print(result)

**Go/TS comparison:** In Go, `go func()` fires and forgets. In Python, calling `greet("Kyle", 1)` returns a **coroutine object** — it doesn't run until you `await` it. Forgetting `await` is the #1 async bug.

In [ ]:
# Experiment: what happens WITHOUT await?
coro = greet("Kyle", 0)
print(f"Type: {type(coro)}")  # <class 'coroutine'>
# The function hasn't run yet! You must await it:
result = await coro
print(f"Result: {result}")

### ✍️ In Your Own Words

What happens when you call an async function without `await`? Why is this different from Go? Write your answer here.

## Concurrent Tasks

**Go/TS comparison:** `gather()` is like `sync.WaitGroup` in Go or `Promise.all()` in TS. `create_task()` is like `go func()`. `as_completed()` is like `select` on channels.

In [ ]:
# asyncio.gather() — run multiple coroutines concurrently
# Like sync.WaitGroup in Go or Promise.all() in TS

async def slow_task(name, delay):
    print(f"  {name} starting...")
    await asyncio.sleep(delay)
    print(f"  {name} done after {delay}s")
    return f"{name} result"

# Sequential — takes 3 seconds
start = time.time()
r1 = await slow_task("A", 1)
r2 = await slow_task("B", 1)
r3 = await slow_task("C", 1)
print(f"Sequential: {time.time() - start:.1f}s\n")

# Concurrent with gather — takes 1 second
start = time.time()
results = await asyncio.gather(
    slow_task("A", 1),
    slow_task("B", 1),
    slow_task("C", 1),
)
print(f"Concurrent: {time.time() - start:.1f}s")
print(f"Results: {results}")

In [ ]:
# create_task() — fire off a task and await later
# Like `go func()` but you get a handle to check on it

async def background_work():
    await asyncio.sleep(2)
    return "background done"

# Start the task (it begins running immediately)
task = asyncio.create_task(background_work())
print("Task started, doing other work...")
await asyncio.sleep(1)
print("Still working...")
# Now wait for it to finish
result = await task
print(f"Task result: {result}")

In [ ]:
# Experiment: as_completed() — process results as they arrive
# Like select on Go channels

async def fetch(name, delay):
    await asyncio.sleep(delay)
    return f"{name} (took {delay}s)"

tasks = [
    asyncio.create_task(fetch("fast", 0.5)),
    asyncio.create_task(fetch("medium", 1.5)),
    asyncio.create_task(fetch("slow", 2.5)),
]

print("Results arrive in completion order:")
for coro in asyncio.as_completed(tasks):
    result = await coro
    print(f"  Got: {result}")

### ✍️ In Your Own Words

When would you use `gather()` vs `create_task()`? Write your answer here.

## Error Handling in Async

In [ ]:
# What happens when a task in gather() raises?
async def might_fail(name, should_fail=False):
    await asyncio.sleep(0.5)
    if should_fail:
        raise ValueError(f"{name} failed!")
    return f"{name} succeeded"

# Default: first exception cancels everything
try:
    results = await asyncio.gather(
        might_fail("A"),
        might_fail("B", should_fail=True),
        might_fail("C"),
    )
except ValueError as e:
    print(f"Caught: {e}")

In [ ]:
# return_exceptions=True — collect errors without crashing
# Like errgroup in Go but errors are returned, not raised
results = await asyncio.gather(
    might_fail("A"),
    might_fail("B", should_fail=True),
    might_fail("C"),
    return_exceptions=True,
)
for r in results:
    if isinstance(r, Exception):
        print(f"  Error: {r}")
    else:
        print(f"  OK: {r}")

### ✍️ In Your Own Words

How does error handling in `gather()` compare to error groups in Go? When would you use `return_exceptions=True`? Write your answer here.

## Async Patterns

In [ ]:
# Async context manager — async with
# Like defer in Go but for async resources

class AsyncTimer:
    def __init__(self, name):
        self.name = name
    
    async def __aenter__(self):
        self.start = time.time()
        print(f"  [{self.name}] Starting...")
        return self
    
    async def __aexit__(self, *args):
        elapsed = time.time() - self.start
        print(f"  [{self.name}] Done in {elapsed:.2f}s")

async with AsyncTimer("my operation"):
    await asyncio.sleep(1)

In [ ]:
# Async generator — async for
# Like a Go channel that yields values over time

async def countdown(n):
    for i in range(n, 0, -1):
        await asyncio.sleep(0.3)
        yield i

print("Countdown:")
async for num in countdown(5):
    print(f"  {num}...")
print("  Go!")

In [ ]:
# Experiment: Semaphore — limit concurrency
# Like a buffered Go channel used as a semaphore

sem = asyncio.Semaphore(2)  # Only 2 tasks run at once

async def limited_task(name):
    async with sem:
        print(f"  {name} started")
        await asyncio.sleep(1)
        print(f"  {name} finished")

start = time.time()
await asyncio.gather(*[limited_task(f"Task-{i}") for i in range(5)])
print(f"Total: {time.time() - start:.1f}s (would be 1s without semaphore)")

### ✍️ In Your Own Words

What's the Python equivalent of a buffered Go channel for concurrency limiting? Write your answer here.

## The Event Loop

**Go/TS comparison:** Node has one event loop you never see. Go has a scheduler you never see. Python **makes you aware** of the event loop — `asyncio.run()` creates one in scripts. Jupyter notebooks already have one running, which is why `await` works directly here.

In [ ]:
# In a .py file, you'd use asyncio.run():
#   async def main():
#       result = await greet("Kyle", 1)
#       print(result)
#   asyncio.run(main())  # Creates event loop, runs main, shuts down

# In Jupyter, the loop is already running:
loop = asyncio.get_event_loop()
print(f"Loop running? {loop.is_running()}")  # True in Jupyter
print(f"This is why await works directly in notebook cells")

### ✍️ In Your Own Words

Why can't you call `asyncio.run()` inside a Jupyter notebook? What does the notebook provide instead? Write your answer here.

## Recap

- ✅ Async is **cooperative** — you must `await` to yield control
- ✅ `await` runs a coroutine — forgetting it is the #1 bug
- ✅ `gather()` for concurrent execution (like `Promise.all()` / `sync.WaitGroup`)
- ✅ `create_task()` for fire-and-await-later (like `go func()`)
- ✅ `return_exceptions=True` for resilient gathering
- ✅ `async with` for async resource management
- ✅ `async for` for async iteration
- ✅ `Semaphore` for concurrency limiting (like buffered Go channels)
- ✅ Notebooks have a running event loop — use `asyncio.run()` only in `.py` files

**Next:** [04_type_hints.ipynb](./04_type_hints.ipynb)